# MHN Ranking Policy Training

This tutorial demonstrates how to train a Modern-Hopfield-style ranking policy network in ``SynPlanner`` and how to fine-tune an already trained MHN checkpoint on a new set of reaction rules and target molecules.


## Basic recommendations

1. Use MHN ranking when you want the policy network to score a runtime set of reaction rules instead of being fixed to one output head per rule.

2. Keep the reaction-rule SMARTS convention stable between rule extraction, MHN training, and planning. The MHN rule representations are built from the paired reaction-rules TSV inferred from the ``*_policy_data.tsv`` file.

3. Start with ``mhn_rule_encoder_type="fingerprint"`` and ``mhn_rule_fp_type="query_cgr"``. Switch to ``mhn_rule_encoder_type="query_cgr_graph"`` only when you intentionally want the GPS rule graph encoder and have enough data/GPU budget.

4. Fine-tuning should use a smaller learning rate than the initial training run. Fine-tuning can add new rules, but it does not change the architecture of the loaded checkpoint; train a fresh checkpoint if you want to change the molecule encoder or rule encoder type.


## 1. Set up input and output data locations


This tutorial uses the rule-extraction outputs produced by Tutorial 03. For a ranking policy, ``SynPlanner`` expects a policy mapping file and a paired reaction-rules TSV with the standard naming convention:

- ``uspto_reaction_rules_policy_data.tsv``
- ``uspto_reaction_rules.tsv``

For fine-tuning, use the same convention for the new rule set, for example:

- ``new_reaction_rules_policy_data.tsv``
- ``new_reaction_rules.tsv``


In [ ]:
import os
from pathlib import Path

results_folder = Path("tutorial_results").resolve()
results_folder.mkdir(exist_ok=True)

# Initial MHN training data from reaction-rule extraction.
policy_data_path = results_folder / "uspto_reaction_rules_policy_data.tsv"
reaction_rules_path = results_folder / "uspto_reaction_rules.tsv"

# New policy data for fine-tuning. The paired rules file should be named
# new_reaction_rules.tsv and placed next to this mapping file.
new_policy_data_path = results_folder / "new_reaction_rules_policy_data.tsv"
new_reaction_rules_path = results_folder / "new_reaction_rules.tsv"

# Output folders.
mhn_policy_network_folder = results_folder / "mhn_ranking_policy_network"
mhn_tuned_policy_network_folder = results_folder / "mhn_ranking_policy_network_tuned"
mhn_policy_network_folder.mkdir(exist_ok=True)
mhn_tuned_policy_network_folder.mkdir(exist_ok=True)

# YAML configs used by the equivalent CLI commands below.
mhn_training_config_path = results_folder / "mhn_ranking_policy_training.yaml"
mhn_tuning_config_path = results_folder / "mhn_ranking_policy_tuning.yaml"


## 2. MHN ranking policy training


### MHN ranking network configuration


In [ ]:
from synplan.utils.config import PolicyNetworkConfig
from synplan.ml.training.supervised import create_policy_dataset, run_policy_training

training_config = PolicyNetworkConfig(
    architecture="mhn_ranking",
    policy_type="ranking",
    embedder_type="gps",  # product/target molecule graph encoder
    vector_dim=512,
    num_conv_layers=5,
    learning_rate=0.0005,
    dropout=0.4,
    num_epoch=100,
    batch_size=1000,
    mhn_association_dim=512,
    mhn_beta=0.05,
    mhn_rule_encoder_type="fingerprint",
    mhn_rule_fp_type="query_cgr",
    mhn_rule_fp_size=2048,
    mhn_rule_fp_min_radius=1,
    mhn_rule_fp_max_radius=4,
    mhn_rule_fp_active_bits=2,
    mhn_normalize_associations=True,
    logger={"type": "csv"},
)

training_config.to_yaml(str(mhn_training_config_path))
mhn_training_config_path


The default configuration above uses QueryCGR rule fingerprints. To train the native QueryCGR graph rule encoder instead, use the same molecule encoder config and change only the rule-side fields:

```python
graph_rule_config = training_config.model_copy(
    update={
        "mhn_rule_encoder_type": "query_cgr_graph",
        "mhn_rule_embedder_type": "gps",
        "mhn_rule_graph_batch_size": 1024,
    }
)
```

The graph rule encoder is opt-in because it is heavier than fingerprints and trains an additional GPS encoder for reaction-rule graphs.


### Creating the MHN ranking training set


The ranking dataset is built from ``policy_data_path``. During MHN training, the paired reaction-rules TSV is inferred from this path and used to build the ordered rule representations.


In [ ]:
datamodule = create_policy_dataset(
    dataset_type="ranking",
    policy_data_path=str(policy_data_path),
    results_dir=str(mhn_policy_network_folder),
    batch_size=training_config.batch_size,
    num_workers=4,
    cache=True,
    accelerator="auto",
)


### Running MHN ranking policy training


<div class="alert alert-warning">
<b>GPU requirement</b>

By default, <code>run_policy_training</code> uses <code>accelerator="gpu"</code>. If you do not have a GPU available, pass <code>accelerator="cpu"</code> or <code>accelerator="auto"</code>.
</div>


In [ ]:
run_policy_training(
    datamodule,
    config=training_config,
    results_path=str(mhn_policy_network_folder),
    accelerator="auto",
)


### Equivalent CLI command


The same initial training run can be launched from the command line:

```bash
synplan ranking_policy_training   --config tutorial_results/mhn_ranking_policy_training.yaml   --policy_data tutorial_results/uspto_reaction_rules_policy_data.tsv   --results_dir tutorial_results/mhn_ranking_policy_network   --workers 4
```


## 3. Fine-tune an already trained MHN checkpoint


Fine-tuning is useful when you have a small new rule set and corresponding target molecules that were not present in the initial policy data. The new mapping file should point labels to the new paired reaction-rules TSV.

For example, if the new mapping file is ``new_reaction_rules_policy_data.tsv``, place ``new_reaction_rules.tsv`` in the same directory. The tuning function uses this paired file to rebuild the ordered rule representations before continuing training.


### Fine-tuning configuration


In [ ]:
tuning_config = training_config.model_copy(
    update={
        "learning_rate": 0.0001,
        "num_epoch": 20,
        "batch_size": 512,
        "logger": {"type": "csv"},
    }
)

tuning_config.to_yaml(str(mhn_tuning_config_path))
mhn_tuning_config_path


### Running fine-tuning from Python


In [ ]:
from synplan.ml.training.supervised import run_mhn_network_tuning

trained_checkpoint_path = mhn_policy_network_folder / "policy_network.ckpt"

run_mhn_network_tuning(
    policy_network_path=str(trained_checkpoint_path),
    new_policy_data_path=str(new_policy_data_path),
    results_path=str(mhn_tuned_policy_network_folder),
    config=tuning_config,
    num_workers=4,
    cache=True,
    accelerator="auto",
)


### Equivalent CLI command


The same fine-tuning run can be launched from the command line:

```bash
synplan mhn_network_tuning   --config tutorial_results/mhn_ranking_policy_tuning.yaml   --policy_network tutorial_results/mhn_ranking_policy_network/policy_network.ckpt   --new_policy_data tutorial_results/new_reaction_rules_policy_data.tsv   --results_dir tutorial_results/mhn_ranking_policy_network_tuned   --workers 4
```

The CLI reads fine-tuning hyperparameters such as ``num_epoch``, ``batch_size``, ``learning_rate``, logger settings, and Trainer options from the YAML config. Path and preprocessing controls remain explicit CLI arguments.


## Results


If the tutorial is executed successfully, the results folder will contain the original rule-extraction outputs, the MHN ranking policy training folder, the fine-tuning config, and the tuned MHN checkpoint folder.


In [ ]:
sorted(Path(results_folder).iterdir(), key=os.path.getmtime, reverse=False)
